In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, OneHotEncoder
from scipy import stats

In [36]:
# Leitura dos Dados
df = pd.read_parquet(r'..\data\processed\bank_features.parquet')

# Configuração para visualizar todas as variávels
pd.set_option('display.max_columns', None)

In [5]:
df.shape

(41188, 27)

In [37]:
# Transformando a variável TARGET (y) em binária ANTES da separação
df['y'] = np.where(df['y'] == 'yes', 1, 0)

In [38]:
# Utilização de 70% para Treino
abt_01, abt_01_test = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['y'] # Devido a desbalanceamento da variável TARGET
)

In [39]:
# Excluindo variável TARGET do treino
abt_01 = abt_01.drop(columns=['y'])

In [40]:
# Validação das bases
print(f'Formato da Memória: {type(abt_01)}')
print(f'Linhas do treino: {abt_01.shape[0]}')
print(f'Linhas do teste: {abt_01_test.shape[0]}')

Formato da Memória: <class 'pandas.DataFrame'>
Linhas do treino: 28831
Linhas do teste: 12357


# Metadados

In [41]:
def visualizar_metadados(df) -> pd.DataFrame:
    """
    Função para visualizar Metadados de um Dataframe.
    return: Tipo, Cardinalidade, Qtd de Nulos e Percent Nulos
    param: Dataframe fornecido para análise
    """

    metadata = pd.DataFrame({
        'Nome': df.columns,
        'Tipo': df.dtypes,
        'Cardinalidade': df.nunique(),
        'Qtd_Nulos': df.isnull().sum(),
        'Percent_Nulos': round((df.isnull().sum() / len(df)) * 100, 2),
    })

    # Função para verificar o teste de Normalidade (Shapiro-Wilk)
    def test_normalidade(series,alpha=0.05):
        if series.dtypes in ['float64', 'int64']:
            statistic, pvalue = stats.shapiro(series.dropna())
            return pvalue > alpha
        else:
            return None

    # Aplicando o teste de Normalidade
    metadata['fl_normal'] = df.apply(test_normalidade)

    metadata = metadata.sort_values(by='Percent_Nulos', ascending=False)
    metadata = metadata.reset_index(drop=True)

    return metadata

In [42]:
visualizar_metadados(abt_01)

c:\Users\felip\PythonProjects\Projetos_Pod_Academy\bank-marketing\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 28831.
  res = hypotest_fun_out(*samples, **kwds)


,Nome,Tipo,Cardinalidade,Qtd_Nulos,Percent_Nulos,fl_normal
0,age,int64,74,0,0.0,False
1,job,str,12,0,0.0,None
2,marital,str,4,0,0.0,None
3,education,str,8,0,0.0,None
4,default,str,3,0,0.0,None
5,housing,str,3,0,0.0,None
6,loan,str,3,0,0.0,None
7,contact,str,2,0,0.0,None
8,month,str,10,0,0.0,None
9,day_of_week,str,5,0,0.0,None


# Tratamento das Variáveis Numéricas

In [43]:
# Copiando para utilizar tratamentos diferentes.
abt_02 = abt_01.copy()
abt_03 = abt_01.copy()

## 1. RobustScaler (Mediana e IQR)

In [44]:
scaler_robust = RobustScaler()

# Selecionando as colunas numéricas
cols_numericas = abt_01.select_dtypes(include=['float64', 'int64', 'int32']).columns

# Escalador Robusto - (Mediana e IQR)
abt_01[cols_numericas] = scaler_robust.fit_transform(abt_01[cols_numericas])
abt_01.shape

(28831, 26)

## 2. Padronização (Média e Desvio Padrão)

In [ ]:
scaler_std = StandardScaler()

# Padronização (Média e Desvio Padrão)
abt_02[cols_numericas] = scaler_std.fit_transform(abt_02[cols_numericas])
abt_02.shape

(28831, 26)

## 3. Normalização (Min-Max)

In [ ]:
scaler_minmax = MinMaxScaler()

# Normalização (Min-Max)
abt_03[cols_numericas] = scaler_minmax.fit_transform(abt_03[cols_numericas])
abt_03.shape

(28831, 26)

# Tratamento das Variáveis Categóricas